<img src="https://raw.githubusercontent.com/AugustoBellanich/poblar/main/docs/img/isotipo.svg" width="48" align="left" style="margin-right:12px">

# poblar · población expuesta a la degradación de tierras

Calcula cuántas personas viven en tierras degradadas a partir de un censo por radio, una cobertura del suelo y un mapa de degradación, y produce los indicadores del OE2 de la CNULD: SO2-1 (pobreza, NBI), SO2-2 (acceso a agua de red) y SO2-3 (población expuesta).

**Cómo se usa:** completar los formularios y ejecutar las celdas en orden (▶ a la izquierda de cada una).

1. **Preparar**: instala poblar y conecta Google Drive.
2. **Período e insumos**: el año y los archivos.
3. **Método**: los criterios estándar del PRAIS 2026 o una variante.
4. **Ejecutar**.
5. **Resultados**: informe, mapas y comparación con la ejecución estándar.

Para mirar una ejecución que ya existe: correr 1, 2 y 3 y pasar directo a los resultados. Si la sesión se corta, volver a correr de 1 a 4: se retoma desde lo que ya quedó guardado en Drive.

In [ ]:
#@title 1 · Preparar { display-mode: "form" }
#@markdown Instala poblar y GDAL y conecta Google Drive (la primera vez pide permiso). Tarda alrededor de un minuto.
VERSION = 'main'  #@param {type:"string"}
#@markdown `main` es la última versión. Para repetir exactamente una ejecución anterior, poner el commit que figura en su informe.

import importlib
import subprocess
import sys

REPO = '/content/poblar_repo'

def _sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f'{cmd}\n{r.stderr or r.stdout}')
    return r.stdout.strip()

_sh(f'rm -rf {REPO} && git clone -q https://github.com/AugustoBellanich/poblar {REPO} '
    f'&& git -C {REPO} checkout -q {VERSION}')
_sh(f'{sys.executable} -m pip install -q -e {REPO}')
if REPO not in sys.path:                      # la instalación editable se ve sin reiniciar
    sys.path.insert(0, REPO)
importlib.invalidate_caches()
for _m in [m for m in sys.modules if m == 'poblar' or m.startswith('poblar.')]:
    del sys.modules[_m]                       # descarta una versión importada antes en esta sesión
_sh('which gdalwarp > /dev/null || (apt-get -qq update && apt-get -qq install -y gdal-bin)')

from google.colab import drive
drive.mount('/content/drive')

from poblar import cuaderno
display(cuaderno.aviso('Listo', f"poblar {_sh(f'git -C {REPO} log -1 --format=%h·%s·%cs')}<br>"
                                f"{_sh('gdalinfo --version')}<br>Google Drive conectado.", 'ok'))

In [ ]:
#@title 2 · Período e insumos { display-mode: "form" }
ANIO = 2022  #@param [2010, 2022] {type:"raw", allow-input: true}
#@markdown ---
#@markdown **Archivos.** Cada uno con su ruta, se llame como se llame; las rutas sin `/` inicial se buscan en `CARPETA_INSUMOS`.
#@markdown Vacío = el archivo del PRAIS 2026 para ese año. `-` = no usarlo (por ejemplo, sin mapa de degradación se calcula la población pero no el cruce ni las tablas).
CARPETA_INSUMOS = '/content/drive/MyDrive/PRAIS/poblar/insumos'  #@param {type:"string"}
TABLA_CENSO = ''  #@param {type:"string"}
RADIOS = ''  #@param {type:"string"}
REGIONES = ''  #@param {type:"string"}
COBERTURA = ''  #@param {type:"string"}
DEGRADACION = ''  #@param {type:"string"}
#@markdown ---
#@markdown Carpeta donde se guardan las ejecuciones (una subcarpeta por ejecución).
SALIDA = '/content/drive/MyDrive/PRAIS/poblar/ejecuciones'  #@param {type:"string"}

from pathlib import Path

ANIO = int(ANIO)
PRAIS = {   # nombres de los archivos del PRAIS 2026
    'tabla_censo': f'tabla_censo{ANIO}.xlsx',
    'radios':      f'radios_{ANIO}.zip',
    'regiones':    'regiones_PRAIS.gpkg',
    'cobertura':   f'MB_prais_{2015 if ANIO == 2010 else 2023}_100m.tif',
    'degradacion': f'degradacion_{"referencia" if ANIO == 2010 else "reporte"}_{ANIO}.tif',
}
_indicados = {'tabla_censo': TABLA_CENSO, 'radios': RADIOS, 'regiones': REGIONES,
              'cobertura': COBERTURA, 'degradacion': DEGRADACION}
INSUMOS, SIN_USAR = {}, []
for _rol, _valor in _indicados.items():
    _valor = _valor.strip() or PRAIS.get(_rol, '')
    if _valor == '-':
        SIN_USAR.append(_rol)
    elif _valor:
        INSUMOS[_rol] = str(Path(CARPETA_INSUMOS) / _valor)   # una ruta absoluta queda igual
SALIDA = Path(SALIDA)

_vista, _ok = cuaderno.revisar_insumos(INSUMOS, SIN_USAR)
display(_vista)

In [ ]:
#@title 3 · Método { display-mode: "form" }
#@markdown **Estándar**: los criterios del PRAIS 2026; reproduce las cifras publicadas y controla que los archivos sean los oficiales.
#@markdown **Personalizado**: cambia alguno de los criterios de abajo. Se guarda como otra ejecución y reutiliza de la estándar del mismo año lo que no cambia.
#@markdown **Archivo propio**: un método escrito a mano (copia de `aplicaciones/prais2026/metodo.yaml`).
METODO = 'Estándar PRAIS 2026'  #@param ["Estándar PRAIS 2026", "Personalizado", "Archivo propio"]
ARCHIVO_METODO = ''  #@param {type:"string"}
#@markdown ---
#@markdown **Criterios** (solo con *Personalizado*; *estándar* es el valor del PRAIS 2026).
#@markdown Resolución de la grilla de trabajo. Cambiarla recalcula todo (varias horas).
RESOLUCION = 'automática, ~80 m (estándar)'  #@param ["automática, ~80 m (estándar)", "100 m", "250 m", "500 m"]
#@markdown Radios que se usan. «Todos los del mapa» suma los radios sin dato censal, con población 0.
RADIOS_USADOS = 'con dato censal y cartografía (estándar)'  #@param ["con dato censal y cartografía (estándar)", "todos los del mapa"]
#@markdown Un radio que no toca ninguna región va a la más cercana si está a esta distancia o menos (km). Estándar: 50.
DISTANCIA_REGION_KM = 50  #@param {type:"integer"}
#@markdown Peso de los píxeles urbanos dentro de radios mixtos y rurales, al repartir la población.
PESO_URBANO = '1 − peso no urbano (estándar)'  #@param ["1 − peso no urbano (estándar)", "1"]
#@markdown Cómo se informan los radios mixtos en el cruce y en las tablas por tipo de radio.
RADIOS_MIXTOS = 'como rurales (estándar)'  #@param ["como rurales (estándar)", "como urbanos"]
#@markdown Urbano/rural en las tablas: por persona (dato del censo) o por tipo de radio. Con la misma definición en país y en tierras degradadas, los porcentajes se pueden comparar entre sí.
URBANO_RURAL_PAIS = 'por persona (estándar)'  #@param ["por persona (estándar)", "por tipo de radio"]
URBANO_RURAL_TD = 'por tipo de radio (estándar)'  #@param ["por tipo de radio (estándar)", "por persona"]
#@markdown Nombre corto de la variante, para la carpeta (vacío = se arma con los cambios).
ETIQUETA = ''  #@param {type:"string"}

import pandas as pd
import yaml
from poblar import config as pcfg
from poblar.ejecutar import etapas_afectadas

METODO_ESTANDAR = Path(REPO) / 'aplicaciones/prais2026/metodo.yaml'
_std = yaml.safe_load(METODO_ESTANDAR.read_text(encoding='utf-8'))
APLICACION = _std['aplicacion']
NOMBRE_ESTANDAR = f'{APLICACION}_{ANIO}'
CAMBIOS, _filas, _codigos = {}, [], []

def _cambio(criterio, estandar, elegido, codigo, claves):
    _filas.append({'Criterio': criterio, 'Estándar': estandar, 'Esta ejecución': elegido})
    _codigos.append(codigo)
    CAMBIOS.update(claves)

if METODO == 'Personalizado':
    if not RESOLUCION.startswith('automática'):
        _m = int(RESOLUCION.split()[0])
        _cambio('Resolución', 'automática (~80 m)', RESOLUCION, f'res{_m}',
                {'grilla_trabajo.modo': 'fija', 'grilla_trabajo.resolucion_m': _m})
    if RADIOS_USADOS == 'todos los del mapa':
        _cambio('Radios', 'con dato censal y cartografía', 'todos los del mapa', 'radios-todos',
                {'radios.union': 'left'})
    if DISTANCIA_REGION_KM != _std['region']['umbral_vecino_km']:
        _cambio('Distancia a región', f"{_std['region']['umbral_vecino_km']} km", f'{DISTANCIA_REGION_KM} km',
                f'dist{DISTANCIA_REGION_KM}', {'region.umbral_vecino_km': DISTANCIA_REGION_KM})
    if PESO_URBANO == '1':
        _cambio('Peso urbano', '1 − peso no urbano', '1', 'peso1', {'dasimetria.ponderadores.peso_urbano': 'uno'})
    if RADIOS_MIXTOS == 'como urbanos':
        _cambio('Radios mixtos', 'rurales', 'urbanos', 'mixtos-urb', {'cruce.ambito_reporte.Mixto': 'Urbano'})
    if URBANO_RURAL_PAIS == 'por tipo de radio':
        _cambio('Urbano/rural, país', 'por persona', 'por tipo de radio', 'ur-pais-radio',
                {'tablas.urbano_rural.pais': 'radio'})
    if URBANO_RURAL_TD == 'por persona':
        _extra = [v for v in ('pobla_total_urbana', 'pobla_total_rural', 'nbi_urbana', 'nbi_rural',
                              'agua_red_urbana', 'agua_red_rural') if v not in _std['cruce']['variables']]
        _cambio('Urbano/rural, tierras degradadas', 'por tipo de radio', 'por persona', 'ur-td-persona',
                {'tablas.urbano_rural.td': 'persona', 'cruce.variables': _std['cruce']['variables'] + _extra})

BASE = None
if METODO == 'Archivo propio':
    METODO_BASE = Path(ARCHIVO_METODO)
    if not METODO_BASE.exists():
        raise FileNotFoundError(f'No se encontró el método: {METODO_BASE}')
    _app = yaml.safe_load(METODO_BASE.read_text(encoding='utf-8')).get('aplicacion', 'propio')
    NOMBRE = ETIQUETA.strip() or f'{_app}_{ANIO}'
    REGISTRO = None
    display(cuaderno.aviso('Método propio', f'<code>{METODO_BASE.name}</code> · ejecución <code>{NOMBRE}</code>.'))
elif CAMBIOS:
    METODO_BASE = METODO_ESTANDAR
    NOMBRE = f"{NOMBRE_ESTANDAR}_{ETIQUETA.strip() or '_'.join(_codigos)}"
    # Del registro oficial se usan solo las huellas de los insumos: la variante no tiene que dar las cifras oficiales
    _reg = Path(REPO) / f'aplicaciones/prais2026/periodos/{ANIO}.yaml'
    REGISTRO = None
    if _reg.exists():
        _r = yaml.safe_load(_reg.read_text(encoding='utf-8'))
        REGISTRO = Path('/content/poblar_metodos') / f'registro_{NOMBRE}.yaml'
        REGISTRO.parent.mkdir(parents=True, exist_ok=True)
        REGISTRO.write_text(yaml.safe_dump({'anio': ANIO, 'insumos': _r.get('insumos', {})}, allow_unicode=True),
                            encoding='utf-8')
    if (SALIDA / NOMBRE_ESTANDAR / '0_insumos').exists():
        BASE = SALIDA / NOMBRE_ESTANDAR
    _recalc = etapas_afectadas(CAMBIOS)
    _reuso = [e for e in ['radios', 'grilla', 'dasimetria', 'cruce', 'tablas'] if e not in _recalc]
    _texto = (f'Ejecución <code>{NOMBRE}</code>. Se recalcula: {", ".join(_recalc)}. '
              + (f'Se reutiliza de <code>{NOMBRE_ESTANDAR}</code>: {", ".join(_reuso)}.' if BASE and _reuso else
                 '' if not _reuso else f'No está la ejecución estándar <code>{NOMBRE_ESTANDAR}</code> para reutilizar: se calcula todo.'))
    display(cuaderno.tabla(pd.DataFrame(_filas)) + cuaderno.aviso('Variante', _texto, 'info'))
else:
    METODO_BASE = METODO_ESTANDAR
    NOMBRE = NOMBRE_ESTANDAR
    _reg = Path(REPO) / f'aplicaciones/prais2026/periodos/{ANIO}.yaml'
    REGISTRO = _reg if _reg.exists() else None
    _txt = 'con el registro del período: huellas de los archivos y cifras publicadas' if REGISTRO else 'sin registro del período'
    if METODO == 'Personalizado':
        _txt += '. No se eligió ningún cambio: es la ejecución estándar'
    display(cuaderno.aviso('Método estándar PRAIS 2026', f'Ejecución <code>{NOMBRE}</code>, {_txt}.'))
CARPETA = SALIDA / NOMBRE

In [ ]:
#@title 4 · Ejecutar { display-mode: "form" }
#@markdown **Etapas**: `todas` corre las que tienen sus insumos y retoma lo que ya esté en la carpeta. También se puede indicar una lista: insumos, radios, grilla, dasimetria, cruce, publicacion, tablas.
ETAPAS = 'todas'  #@param ["todas", "insumos, radios", "cruce, tablas"] {allow-input: true}
#@markdown **Publicar** las capas en la grilla de entrega (EPSG:4326, nombres de la plataforma). Es la etapa más lenta; para probar criterios no hace falta.
PUBLICAR = True  #@param {type:"boolean"}

from poblar import ejecutar, informe
from poblar.trazabilidad import ControlFallido

_metodo = METODO_BASE
if CAMBIOS or not PUBLICAR:
    _metodo = pcfg.variar(METODO_BASE, CAMBIOS, Path('/content/poblar_metodos') / f'metodo_{NOMBRE}.yaml',
                          quitar=[] if PUBLICAR else ['publicacion'])
_etapas = 'todas' if ETAPAS.strip() == 'todas' else [e.strip() for e in ETAPAS.split(',') if e.strip()]
try:
    ejecutar(_metodo, anio=ANIO, insumos=INSUMOS, registro=REGISTRO, salida=SALIDA, etapas=_etapas,
             trabajo='/content/trabajo', nombre=NOMBRE, base=BASE)
except ControlFallido as e:
    print(e)
_ctrl = pd.read_csv(CARPETA / 'controles.csv') if (CARPETA / 'controles.csv').exists() else None
_clase, _estado, _det = informe._estado(_ctrl)
display(cuaderno.aviso(_estado, f'{_det}<br>Carpeta: <code>{CARPETA}</code>', _clase))

## Resultados

El **informe** reúne lo principal de la ejecución: el estado de los controles por etapa, las cifras en tierras degradadas, mapas de vista rápida y las tablas de indicadores. Se guarda en la carpeta de la ejecución (`informe_<nombre>.html`) y se puede abrir en el navegador o compartir.

In [ ]:
#@title 5 · Informe { display-mode: "form" }
#@markdown Volver a generarlo con la versión instalada de poblar (por ejemplo, para una ejecución hecha con una versión anterior). Tarda uno o dos minutos por los mapas.
REGENERAR = False  #@param {type:"boolean"}
from poblar import informe

_ruta = informe.ruta(CARPETA)
if REGENERAR or not _ruta.exists():
    _ruta = informe.generar(CARPETA)
display(cuaderno.Html(_ruta.read_text(encoding='utf-8')))

In [ ]:
#@title 6 · Explorar un producto { display-mode: "form" }
#@markdown Mapa de vista rápida, del país o de una región. **expuesta**: población en tierras degradadas; **poblacion**: población por píxel; **degradacion** y **cobertura**: las capas de entrada en la grilla de trabajo; también cualquier variable (nbi, agua_red, pobla_mujer...).
PRODUCTO = 'expuesta'  #@param ["expuesta", "poblacion", "degradacion", "cobertura", "nbi", "agua_red"] {allow-input: true}
REGION = '(todo el país)'  #@param ["(todo el país)", "Altos Andes, Monte y Puna", "Chaco", "Pampa", "Patagonia", "Selva Paranaense"] {allow-input: true}
#@markdown Detalle de la vista, en píxeles de ancho (más es más lento).
ANCHO = 900  #@param {type:"slider", min:400, max:2000, step:100}
import matplotlib.pyplot as plt
from poblar import vista

_disp = vista.productos(CARPETA)
if PRODUCTO not in _disp:
    display(cuaderno.aviso('Producto no disponible', 'En esta ejecución: ' + ', '.join(_disp) + '.', 'adv'))
else:
    vista.mapa(CARPETA, PRODUCTO, region=None if REGION.startswith('(') else REGION, ancho_px=ANCHO)
    plt.show()

In [ ]:
#@title 7 · Comparar con la ejecución estándar { display-mode: "form" }
#@markdown Para una variante: qué indicadores cambian respecto de la ejecución estándar del mismo año. *n*: personas; los porcentajes, en puntos.
REGION_COMPARAR = 'NACIONAL'  #@param ["NACIONAL", "Altos Andes, Monte y Puna", "Chaco", "Pampa", "Patagonia", "Selva Paranaense"] {allow-input: true}
from poblar.tablas import comparar

_std_c = SALIDA / NOMBRE_ESTANDAR
if CARPETA == _std_c:
    display(cuaderno.aviso('Es la ejecución estándar', 'Elegir una variante en el paso 3 para comparar.', 'info'))
elif not (_std_c / '6_tablas').exists() or not (CARPETA / '6_tablas').exists():
    display(cuaderno.aviso('Faltan tablas', f'Hace falta la etapa de tablas en <code>{NOMBRE_ESTANDAR}</code> y en <code>{NOMBRE}</code>.', 'adv'))
else:
    _c = comparar(_std_c, CARPETA)
    _c = _c[_c['region'] == REGION_COMPARAR]
    if _c.empty:
        display(cuaderno.aviso('Sin diferencias', f'Los indicadores de {REGION_COMPARAR} son iguales.', 'ok'))
    else:
        _c = _c.assign(ambito=_c['ambito'].map({'pais': 'País', 'td': 'Tierras degradadas'}).fillna(_c['ambito']))
        _t = _c[['ambito', 'variable', 'grupo', 'n_a', 'n_b', 'dif_n', 'incidencia_a', 'incidencia_b',
                 'exposicion_a', 'exposicion_b']].rename(columns={
            'ambito': 'Ámbito', 'variable': 'Variable', 'grupo': 'Grupo', 'n_a': 'n estándar', 'n_b': 'n variante',
            'dif_n': 'Diferencia', 'incidencia_a': 'Incidencia estándar (%)', 'incidencia_b': 'Incidencia variante (%)',
            'exposicion_a': 'Exposición estándar (%)', 'exposicion_b': 'Exposición variante (%)'})
        display(cuaderno.tabla(_t, numeros={'n estándar': 0, 'n variante': 0, 'Diferencia': 0,
                                            'Incidencia estándar (%)': 2, 'Incidencia variante (%)': 2,
                                            'Exposición estándar (%)': 2, 'Exposición variante (%)': 2},
                               titulo=f'{NOMBRE} frente a {NOMBRE_ESTANDAR} · {REGION_COMPARAR}'))

In [ ]:
#@title 8 · Tablas de dos períodos { display-mode: "form" }
#@markdown Une las tablas de indicadores de 2010 y 2022 con una columna por período. Hace falta la etapa de tablas en los dos años.
#@markdown Variante a unir (vacío = la estándar); por ejemplo `mixtos-urb`.
VARIANTE = ''  #@param {type:"string"}
from poblar.tablas import unir_periodos

_suf = f'_{VARIANTE.strip()}' if VARIANTE.strip() else ''
_ejec = {a: SALIDA / f'{APLICACION}_{a}{_suf}' for a in (2010, 2022)}
_dest = SALIDA / f'{APLICACION}_tablas{_suf}'
for _nombre, _t in unir_periodos(_ejec, _dest).items():
    display(cuaderno.tabla(_t.reset_index() if _t.index.name else _t, titulo=_nombre))
display(cuaderno.aviso('Tablas guardadas', f'<code>{_dest}</code>', 'ok'))